<a href="https://colab.research.google.com/github/pointernullbrain/Iris-Classification/blob/main/P167590_STQD6324_IrisClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
from ucimlrepo import fetch_ucirepo
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier, LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

In [11]:
#Create sparksession
spark = SparkSession.builder \
    .appName("IrisClassification_Assignment1") \
    .getOrCreate()

In [2]:
#Fetch dataset from ucirepo
iris = fetch_ucirepo(id=53)

In [12]:
#Dataset is split into 2: Targets and features
#Features contain the sepal length, width and petal length, width
#Target contains the species
#Convert target and feature to dataframe, then merge by column. create pandas dataframe first
iris_features = pd.DataFrame(iris.data.features)
iris_targets = pd.DataFrame(iris.data.targets)
pandas_df = pd.concat([iris_features, iris_targets], axis=1)
pandas_df.head()

,sepal length,sepal width,petal length,petal width,class
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [13]:
#Convert pandas dataframe to spark dataframe
iris_df = spark.createDataFrame(pandas_df)
iris_df.show(5)

+------------+-----------+------------+-----------+-----------+
|sepal length|sepal width|petal length|petal width|      class|
+------------+-----------+------------+-----------+-----------+
|         5.1|        3.5|         1.4|        0.2|Iris-setosa|
|         4.9|        3.0|         1.4|        0.2|Iris-setosa|
|         4.7|        3.2|         1.3|        0.2|Iris-setosa|
|         4.6|        3.1|         1.5|        0.2|Iris-setosa|
|         5.0|        3.6|         1.4|        0.2|Iris-setosa|
+------------+-----------+------------+-----------+-----------+
only showing top 5 rows


In [14]:
#String indexer to convert categorical class value to numerical label indices
indexer = StringIndexer(inputCol="class", outputCol="label")
iris_df_indexed = indexer.fit(iris_df).transform(iris_df)
iris_df_indexed.show(5)

+------------+-----------+------------+-----------+-----------+-----+
|sepal length|sepal width|petal length|petal width|      class|label|
+------------+-----------+------------+-----------+-----------+-----+
|         5.1|        3.5|         1.4|        0.2|Iris-setosa|  0.0|
|         4.9|        3.0|         1.4|        0.2|Iris-setosa|  0.0|
|         4.7|        3.2|         1.3|        0.2|Iris-setosa|  0.0|
|         4.6|        3.1|         1.5|        0.2|Iris-setosa|  0.0|
|         5.0|        3.6|         1.4|        0.2|Iris-setosa|  0.0|
+------------+-----------+------------+-----------+-----------+-----+
only showing top 5 rows


In [16]:
#Vectorassembler to convert column features to singular vector
feature_cols = ["sepal length","sepal width","petal length","petal width"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
iris_df_final = assembler.transform(iris_df_indexed)
iris_df_final.show(5)

+------------+-----------+------------+-----------+-----------+-----+-----------------+
|sepal length|sepal width|petal length|petal width|      class|label|         features|
+------------+-----------+------------+-----------+-----------+-----+-----------------+
|         5.1|        3.5|         1.4|        0.2|Iris-setosa|  0.0|[5.1,3.5,1.4,0.2]|
|         4.9|        3.0|         1.4|        0.2|Iris-setosa|  0.0|[4.9,3.0,1.4,0.2]|
|         4.7|        3.2|         1.3|        0.2|Iris-setosa|  0.0|[4.7,3.2,1.3,0.2]|
|         4.6|        3.1|         1.5|        0.2|Iris-setosa|  0.0|[4.6,3.1,1.5,0.2]|
|         5.0|        3.6|         1.4|        0.2|Iris-setosa|  0.0|[5.0,3.6,1.4,0.2]|
+------------+-----------+------------+-----------+-----------+-----+-----------------+
only showing top 5 rows


In [17]:
#Select only features and labels column from the dataset
iris_df_model = iris_df_final.select("features","label")

In [18]:
#Split data into train and test: 80% train 20% test
train_data, test_data = iris_df_model.randomSplit([0.8,0.2], seed = 123)
print(f"Training Data Count: {train_data.count()}")
print(f"Testing Data Count: {test_data.count()}")

Training Data Count: 122
Testing Data Count: 28


In [19]:
#Define the performance evaluators
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
evaluator_prec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
evaluator_rec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")

#Function to print metrics cleanly
def print_metrics(model_name, predictions):
    acc = evaluator_acc.evaluate(predictions)
    f1 = evaluator_f1.evaluate(predictions)
    prec = evaluator_prec.evaluate(predictions)
    rec = evaluator_rec.evaluate(predictions)
    print(f"--- {model_name} Performance ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}\n")

In [20]:
#Decision Tree model
dt = DecisionTreeClassifier(labelCol="label", featuresCol="features", seed=42)

#Grid Search parameters
#Gini - how frequent randomly chosen element will be labelled incorrectly
#Entropy - Reduction in uncertainty after split
#Maxdepth - 3 (frequent underfit), 10 (frequent overfit), 5 (middle)
dt_paramGrid = (ParamGridBuilder()
             .addGrid(dt.maxDepth, [3, 5, 10])
             .addGrid(dt.impurity, ['gini', 'entropy'])
             .build())

#numFolds = 3 meaning it splits train data into 3, test on set 1,2 | 1,3 | 2,3 and take best
dt_cv = CrossValidator(estimator=dt, estimatorParamMaps=dt_paramGrid,
                       evaluator=evaluator_acc, numFolds=3, seed=42)

#Train and predict
print("Training Decision Tree...")
dt_cvModel = dt_cv.fit(train_data)
dt_predictions = dt_cvModel.transform(test_data)
print_metrics("Decision Tree", dt_predictions)

Training Decision Tree...
--- Decision Tree Performance ---
Accuracy:  1.0000
F1-Score:  1.0000
Precision: 1.0000
Recall:    1.0000



In [21]:
#Random forest
rf = RandomForestClassifier(labelCol="label", featuresCol="features", seed=42)

# Grid Search Parameters
#More trees = more computing, less variance. Cap trees at 30, assume it will plateau by then
#maxDepth 3 or 5 - prevent tree from "overthinking"
rf_paramGrid = (ParamGridBuilder()
             .addGrid(rf.numTrees, [10, 20, 30])
             .addGrid(rf.maxDepth, [3, 5])
             .build())

rf_cv = CrossValidator(estimator=rf, estimatorParamMaps=rf_paramGrid,
                       evaluator=evaluator_acc, numFolds=3, seed=42)

#Train and predict
print("Training Random Forest...")
rf_cvModel = rf_cv.fit(train_data)
rf_predictions = rf_cvModel.transform(test_data)
print_metrics("Random Forest", rf_predictions)

Training Random Forest...
--- Random Forest Performance ---
Accuracy:  1.0000
F1-Score:  1.0000
Precision: 1.0000
Recall:    1.0000



In [22]:
#Logistic regression
lr = LogisticRegression(labelCol="label", featuresCol="features", family="multinomial")

# Grid Search Parameters
#regParam - 0.01 (overfit risk), 1.0 (underfit risk). prevents model from assigning too much importance on a single feature
#elasticNetParam:
  # 0.0 - shrink feature coefficient uniformly, but not to 0, ideal if structural measurement are partially important
  # 1.0 - forces coefficient of less useful features to 0, feature selection
  # 0.5 - combine both

lr_paramGrid = (ParamGridBuilder()
             .addGrid(lr.regParam, [0.01, 0.1, 1.0])
             .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
             .addGrid(lr.maxIter, [10, 50])
             .build())

lr_cv = CrossValidator(estimator=lr, estimatorParamMaps=lr_paramGrid,
                       evaluator=evaluator_acc, numFolds=3, seed=42)

#Train and predict
print("Training Logistic Regression...")
lr_cvModel = lr_cv.fit(train_data)
lr_predictions = lr_cvModel.transform(test_data)
print_metrics("Logistic Regression", lr_predictions)

Training Logistic Regression...
--- Logistic Regression Performance ---
Accuracy:  1.0000
F1-Score:  1.0000
Precision: 1.0000
Recall:    1.0000

